# Analyse Segmentée par Canal - Brief 2 Module 3
Ce notebook analyse les données issues des trois canaux d'entrée : Email, Web et Chat.

In [ ]:
import sys
import os
from pathlib import Path

# Ajout du répertoire racine au sys.path pour permettre les imports du module 'src'
root_path = str(Path(os.getcwd()))
if root_path not in sys.path:
    sys.path.append(root_path)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy
from src.storage.load import get_engine

# Configuration des graphiques
%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
# Chargement des données directement via SQLAlchemy pour l'analyse
try:
    engine = get_engine()
    df = pd.read_sql("SELECT * FROM demandes", con=engine)
    print(f"Total des lignes chargées : {len(df)}")
    if not df.empty:
        display(df.head())
except Exception as e:
    print(f"Erreur lors du chargement des données : {e}")

In [ ]:
if 'df' in locals() and not df.empty:
    # Distribution des canaux
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='canal', palette='viridis')
    plt.title('Nombre de demandes par canal')
    plt.show()

    # Longueur des messages par canal
    df['input_length'] = df['input_text'].str.len()
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='canal', y='input_length')
    plt.title('Longueur des messages par canal')
    plt.show()

    print("Moyenne des longueurs par canal :")
    print(df.groupby('canal')['input_length'].mean())
else:
    print("DataFrame 'df' non disponible ou vide.")

In [ ]:
if 'df' in locals() and not df.empty:
    # État de la déduplication
    print("Répartition du statut de déduplication :")
    if 'dedup_status' in df.columns:
        print(df['dedup_status'].fillna('original').value_counts())
    else:
        print("La colonne 'dedup_status' n'est pas présente dans le DataFrame.")

    # Aperçu des métadonnées
    print("\nExemple de métadonnées canal (Web) :")
    web_data = df[df['canal'] == 'web']
    if not web_data.empty:
        import json
        metadata = web_data['canal_metadata'].iloc[0]
        if isinstance(metadata, str):
            print(json.loads(metadata))
        else:
            print(metadata)
    else:
        print("Aucune donnée 'web' trouvée pour afficher les métadonnées.")
else:
    print("DataFrame 'df' non disponible ou vide.")